In [4]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import CRUD module
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Update with your username and password
username = "aacuser"
password = "password123"  # Remplace par ton mot de passe

# Connect to database via CRUD Module
db = AnimalShelter()

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invalid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here.
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Add Grazioso Salvare's logo
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# Define filter options
filter_options = [
    {'label': 'No Filter (Show All)', 'value': 'all'},
    {'label': 'Water Rescue', 'value': 'water'},
    {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
    {'label': 'Disaster or Individual Tracking', 'value': 'disaster'}
]

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display':'none'}),
    
    # Header with Logo and Title
    html.Div([
        html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                 style={'height': '80px', 'float': 'left', 'margin': '10px'}),
        html.Center(html.B(html.H1('CS-340 Dashboard - Jean Lukenson Collin', 
                                   style={'paddingTop': '20px'}))),
    ]),
    
    html.Hr(),
    
    # Interactive Filtering Options
    html.Div([
        html.H3("Select Rescue Type:", style={'textAlign': 'center'}),
        html.Div(
            dcc.RadioItems(
                id='filter-type',
                options=filter_options,
                value='all',
                labelStyle={'display': 'inline-block', 'margin': '10px'},
                style={'textAlign': 'center', 'fontSize': '18px'}
            ),
            style={'textAlign': 'center'}
        ),
    ]),
    
    html.Hr(),
    
    # Interactive Data Table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        page_current=0,
        page_action='native',
        sort_action='native',
        sort_mode='multi',
        filter_action='native',
        row_selectable='single',
        selected_rows=[0],
        style_table={'overflowX': 'auto', 'maxHeight': '400px'},
        style_cell={
            'textAlign': 'left',
            'minWidth': '50px',
            'maxWidth': '200px',
            'overflow': 'hidden',
            'textOverflow': 'ellipsis'
        },
        style_header={
            'backgroundColor': '#2c3e50',
            'color': 'white',
            'fontWeight': 'bold',
            'textAlign': 'center'
        },
        style_data_conditional=[
            {
                'if': {'row_index': 'odd'},
                'backgroundColor': '#f8f9fa'
            }
        ]
    ),
    
    html.Br(),
    html.Div(id='selected-row-info', style={'textAlign': 'center', 'fontSize': '16px', 'margin': '10px'}),
    html.Hr(),
    
    # Charts Side-by-Side
    html.Div(className='row',
             style={'display': 'flex', 'flexDirection': 'row'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',
            style={'width': '50%', 'padding': '10px'}
        ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            style={'width': '50%', 'height': '500px', 'padding': '10px'}
        )
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################

# This callback filters the data table based on the selected rescue type
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    """
    Filter the data based on the selected rescue type.
    Uses the CRUD module to query MongoDB with the appropriate filter.
    """
    query = {}
    
    if filter_type == 'all':
        query = {}
    elif filter_type == 'water':
        # Water Rescue: dogs, age <= 2 years (104 weeks), 
        # breeds: Labrador Retriever, Newfoundland, Golden Retriever
        water_breeds = ['Labrador Retriever', 'Newfoundland', 'Golden Retriever']
        query = {
            'animal_type': 'Dog',
            'age_upon_outcome_in_weeks': {'$lte': 104},
            'breed': {'$in': water_breeds}
        }
    elif filter_type == 'mountain':
        # Mountain or Wilderness Rescue: dogs, age <= 2 years,
        # breeds: German Shepherd, Belgian Malinois, Siberian Husky
        mountain_breeds = ['German Shepherd', 'Belgian Malinois', 'Siberian Husky']
        query = {
            'animal_type': 'Dog',
            'age_upon_outcome_in_weeks': {'$lte': 104},
            'breed': {'$in': mountain_breeds}
        }
    elif filter_type == 'disaster':
        # Disaster or Individual Tracking: dogs, age <= 2 years,
        # breeds: German Shepherd, Belgian Malinois, Golden Retriever
        disaster_breeds = ['German Shepherd', 'Belgian Malinois', 'Golden Retriever']
        query = {
            'animal_type': 'Dog',
            'age_upon_outcome_in_weeks': {'$lte': 104},
            'breed': {'$in': disaster_breeds}
        }
    
    # Query MongoDB using the CRUD module
    results = db.read(query)
    
    # Convert to DataFrame and then to dictionary for the datatable
    if len(results) > 0:
        dff = pd.DataFrame.from_records(results)
        if '_id' in dff.columns:
            dff.drop(columns=['_id'], inplace=True)
        return dff.to_dict('records')
    else:
        return []

# This callback updates the pie chart based on the data table content
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):
    """
    Create a pie chart showing the distribution of breeds in the filtered data.
    """
    if viewData is None or len(viewData) == 0:
        return html.Div("No data available to display", style={'textAlign': 'center', 'padding': '50px'})
    
    # Convert the view data to a pandas DataFrame
    dff = pd.DataFrame.from_dict(viewData)
    
    # Count the breeds
    breed_counts = dff['breed'].value_counts().head(10)
    
    # Create the pie chart
    fig = px.pie(
        breed_counts,
        values=breed_counts.values,
        names=breed_counts.index,
        title='Top 10 Breeds in Filtered Data',
        color_discrete_sequence=px.colors.qualitative.Set3
    )
    
    fig.update_layout(
        height=400,
        margin={'l': 20, 'r': 20, 't': 40, 'b': 20}
    )
    
    return [dcc.Graph(figure=fig)]

# This callback highlights a row on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns:
        return [{
            'if': {'column_id': i},
            'background_color': '#D2F3FF'
        } for i in selected_columns]
    return []

# This callback displays info about the selected row
@app.callback(
    Output('selected-row-info', 'children'),
    [Input('datatable-id', 'selected_rows'),
     Input('datatable-id', 'data')]
)
def update_selected_row_info(selected_rows, table_data):
    if selected_rows and len(selected_rows) > 0 and len(table_data) > 0:
        row_index = selected_rows[0]
        if row_index < len(table_data):
            row_data = table_data[row_index]
            name = row_data.get('name', 'Unknown')
            breed = row_data.get('breed', 'Unknown')
            outcome = row_data.get('outcome_type', 'Unknown')
            return f"Selected: {name} - {breed} ({outcome})"
    return "No row selected"

# This callback updates the geolocation chart for the selected data entry
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    """
    Update the geolocation map based on the selected row in the data table.
    """
    if viewData is None or len(viewData) == 0:
        return html.Div("No data available to display on map", 
                       style={'textAlign': 'center', 'padding': '50px'})
    
    dff = pd.DataFrame.from_dict(viewData)
    
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]
    
    # Make sure the row index is valid
    if row >= len(dff):
        row = 0
    
    # Try to find latitude and longitude columns by name
    lat_col = None
    lon_col = None
    name_col = None
    breed_col = None
    
    for col in dff.columns:
        if 'lat' in col.lower():
            lat_col = col
        if 'lon' in col.lower() or 'long' in col.lower():
            lon_col = col
        if 'name' in col.lower():
            name_col = col
        if 'breed' in col.lower():
            breed_col = col
    
    # Get the selected row data
    try:
        lat = float(dff.iloc[row][lat_col]) if lat_col and pd.notna(dff.iloc[row][lat_col]) else 30.75
        lon = float(dff.iloc[row][lon_col]) if lon_col and pd.notna(dff.iloc[row][lon_col]) else -97.48
        animal_name = str(dff.iloc[row][name_col]) if name_col and pd.notna(dff.iloc[row][name_col]) else "Unknown"
        breed = str(dff.iloc[row][breed_col]) if breed_col and pd.notna(dff.iloc[row][breed_col]) else "Unknown"
    except (ValueError, IndexError, KeyError) as e:
        lat = 30.75
        lon = -97.48
        animal_name = "Unknown"
        breed = "Unknown"
    
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '100%', 'height': '450px'},
               center=[lat, lon],
               zoom=12,
               children=[
                   dl.TileLayer(id="base-layer-id"),
                   dl.Marker(position=[lat, lon],
                             children=[
                                 dl.Tooltip(breed),
                                 dl.Popup([
                                     html.H1("Animal Name"),
                                     html.P(animal_name),
                                     html.Hr(),
                                     html.P(f"Breed: {breed}"),
                                     html.P(f"Location: {lat}, {lon}")
                                 ])
                             ])
               ])
    ]

# Run app and display result in jupyterlab mode
app.run_server()

Dash app running on https://slowsimple-floodpresto-3000.codio.io/proxy/8050/
